In [ ]:
!pip install -q -U transformers accelerate
!pip install -q peft trl bitsandbytes datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 34.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

filepath = "/content/drive/MyDrive/Colab Notebooks/output.jsonl"

data = []
with open(filepath, "r") as f:
    for line in f:
        try:
            data.append(json.loads(line))
        except json.JSONDecodeError:
            continue

df = pd.DataFrame(data)
print(f"Total loaded: {len(df)}")
print(df['label'].value_counts())

Total loaded: 74184
label
benign      39284
phishing    34900
Name: count, dtype: int64


In [ ]:
SYSTEM_PROMPT = """You are a phishing detection model. You analyze structured data extracted from web pages and determine whether a URL is benign or phishing.

You will receive a JSON object containing:
- url: parsed URL features including scheme, hostname, registered domain, detected brands, and brand-domain mismatches
- redirects: redirect chain analysis including cross-domain redirects and domain changes
- html: page content including title, visible text, forms with input fields, anchors, and iframes
- http: HTTP response metadata including status code, content type, and server

Analyze all the provided data holistically. Consider URL structure, page content, form behavior, redirect patterns, and any brand impersonation indicators.

Respond in this exact format:
Analysis:
- [observation 1]
- [observation 2]
- ...
Label: [benign or phishing]"""


def build_user_prompt(row):
    """Build the user message from raw features (no signals — model must learn to derive them)."""
    features = {
        "url": row["url"],
        "redirects": row["redirects"],
        "html": row["html"],
        "http": row["http"],
    }
    return json.dumps(features, indent=2, default=str)


def build_assistant_response(row):
    """Build the target response using signal statements as the analysis."""
    signals = row["signals"]
    label = row["label"]

    # Build analysis from signal statements
    analysis_lines = []
    for sig in signals:
        analysis_lines.append(f"- {sig['statement']}")

    # If no signals, add a generic observation
    if not analysis_lines:
        analysis_lines.append("- No significant indicators were detected in the page data.")

    analysis = "\n".join(analysis_lines)
    return f"Analysis:\n{analysis}\nLabel: {label}"


def format_example(row):
    """Create the full chat-format training example."""
    return {
        "system": SYSTEM_PROMPT,
        "user": build_user_prompt(row),
        "assistant": build_assistant_response(row),
        "label": row["label"],
    }


# Test with one example
sample = df.iloc[0]
example = format_example(sample)
print("=== SYSTEM ===")
print(example["system"][:200], "...")
print("\n=== USER (first 500 chars) ===")
print(example["user"][:500], "...")
print("\n=== ASSISTANT ===")
print(example["assistant"])

=== SYSTEM ===
You are a phishing detection model. You analyze structured data extracted from web pages and determine whether a URL is benign or phishing.

You will receive a JSON object containing:
- url: parsed UR ...

=== USER (first 500 chars) ===
{
  "url": {
    "raw": "https://ncls1.com/",
    "scheme": "https",
    "hostname": "ncls1.com",
    "registered_domain": "ncls1.com",
    "path": "/",
    "query": "",
    "uses_http_not_https": false,
    "uses_ip_address": false,
    "contains_suspicious_terms": [],
    "detected_brands": [],
    "brand_domain_mismatches": []
  },
  "redirects": {
    "requested_url": "https://ncls1.com/",
    "final_url": "https://ncls1.com/",
    "redirect_count": 0,
    "chain": [],
    "initial_registere ...

=== ASSISTANT ===
Analysis:
- No urgent, security-incident, or account-verification language was detected in the visible text.
- No password, email, or phone input fields were detected in the page forms.
- No cross-domain redirect behavior wa

In [ ]:
# Use 20,000 samples: 10k benign + 10k phishing (balanced)
N_PER_CLASS = 10000

benign_df = df[df['label'] == 'benign'].sample(n=N_PER_CLASS, random_state=42)
phishing_df = df[df['label'] == 'phishing'].sample(n=N_PER_CLASS, random_state=42)
subset = pd.concat([benign_df, phishing_df]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Subset: {len(subset)}")
print(subset['label'].value_counts())

# Split: 90% train, 10% test
train_df, test_df = train_test_split(subset, test_size=0.1, stratify=subset['label'], random_state=42)
print(f"\nTrain: {len(train_df)} | Test: {len(test_df)}")
print(f"Train:\n{train_df['label'].value_counts()}")
print(f"Test:\n{test_df['label'].value_counts()}")

# Format all examples
train_examples = [format_example(row) for _, row in train_df.iterrows()]
test_examples = [format_example(row) for _, row in test_df.iterrows()]
print(f"\n✅ Train examples: {len(train_examples)} | Test examples: {len(test_examples)}")

Subset: 20000
label
phishing    10000
benign      10000
Name: count, dtype: int64

Train: 18000 | Test: 2000
Train:
label
benign      9000
phishing    9000
Name: count, dtype: int64
Test:
label
benign      1000
phishing    1000
Name: count, dtype: int64

✅ Train examples: 18000 | Test examples: 2000


In [ ]:
import torch
import gc
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Store all results
all_results = {}


def cleanup():
    """Free GPU memory between models."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("🧹 GPU memory cleared")


def load_model_and_tokenizer(model_name):
    """Load a model in 4-bit quantization."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    return model, tokenizer


def run_inference(model, tokenizer, examples, max_new_tokens=300, batch_desc="Evaluating"):
    """Run inference on test examples and extract predictions."""
    model.eval()
    predictions = []
    raw_outputs = []

    for i, ex in enumerate(examples):
        if i % 50 == 0:
            print(f"  {batch_desc}: {i}/{len(examples)}...")

        messages = [
            {"role": "system", "content": ex["system"]},
            {"role": "user", "content": ex["user"]},
        ]

        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=3072).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=tokenizer.pad_token_id,
            )

        # Decode only the new tokens
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        raw_outputs.append(response)

        # Extract label from response
        response_lower = response.lower()
        if "label: phishing" in response_lower or "label:phishing" in response_lower:
            predictions.append("phishing")
        elif "label: benign" in response_lower or "label:benign" in response_lower:
            predictions.append("benign")
        elif "phishing" in response_lower.split("\n")[-1]:
            predictions.append("phishing")
        elif "benign" in response_lower.split("\n")[-1]:
            predictions.append("benign")
        else:
            predictions.append("unknown")

    return predictions, raw_outputs


def evaluate_predictions(true_labels, predictions, model_name, phase):
    """Calculate and display metrics."""
    # Filter out unknowns for metrics
    valid = [(t, p) for t, p in zip(true_labels, predictions) if p != "unknown"]
    unknown_count = len(true_labels) - len(valid)

    if valid:
        t_valid, p_valid = zip(*valid)
    else:
        print(f"⚠️ No valid predictions for {model_name} ({phase})")
        return None

    metrics = {
        "model": model_name,
        "phase": phase,
        "accuracy": accuracy_score(t_valid, p_valid),
        "precision": precision_score(t_valid, p_valid, pos_label="phishing", zero_division=0),
        "recall": recall_score(t_valid, p_valid, pos_label="phishing", zero_division=0),
        "f1": f1_score(t_valid, p_valid, pos_label="phishing", zero_division=0),
        "unknown": unknown_count,
        "total": len(true_labels),
    }

    print(f"\n{'='*60}")
    print(f"📊 {model_name} — {phase}")
    print(f"{'='*60}")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1 Score:  {metrics['f1']:.4f}")
    print(f"  Unknown:   {unknown_count}/{len(true_labels)}")
    print(f"\n{classification_report(t_valid, p_valid, digits=4)}")
    return metrics

In [ ]:
def fine_tune_model(model_name, short_name, train_examples, tokenizer_ref=None):
    """Fine-tune a model with LoRA and return the model + tokenizer."""
    print(f"\n{'#'*60}")
    print(f"# FINE-TUNING: {short_name}")
    print(f"{'#'*60}")

    model, tokenizer = load_model_and_tokenizer(model_name)

    # Format training data with chat template
    def format_for_training(ex):
        messages = [
            {"role": "system", "content": ex["system"]},
            {"role": "user", "content": ex["user"]},
            {"role": "assistant", "content": ex["assistant"]},
        ]
        return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

    train_formatted = [format_for_training(ex) for ex in train_examples]
    train_dataset = Dataset.from_list(train_formatted)

    # Prepare for LoRA
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)

    trainable, total = model.get_nb_trainable_parameters()
    print(f"  Total params:     {total:>12,}")
    print(f"  Trainable params: {trainable:>12,} ({100 * trainable / total:.2f}%)")

    # Training config
    training_args = SFTConfig(
        output_dir=f"/content/{short_name}-finetuned",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        logging_steps=50,
        save_strategy="no",
        bf16=True,
        dataset_text_field="text",
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
    )

    print(f"\n🚀 Training {short_name}...")
    start = time.time()
    trainer.train()
    train_time = time.time() - start
    print(f"✅ {short_name} training complete in {train_time/60:.1f} minutes")

    return model, tokenizer, train_time

In [ ]:
# TODO: Add your Hugging Face login here before running this notebook.
# from huggingface_hub import login
# login(token="YOUR_HUGGINGFACE_TOKEN_HERE")

In [ ]:
BASELINE_N = 300
baseline_test = test_examples[:BASELINE_N]
true_labels_baseline = [ex["label"] for ex in baseline_test]

short_name = "Llama-3.2-3B"
model_name = "meta-llama/Llama-3.2-3B-Instruct"

print(f"\n{'#'*60}")
print(f"# BASELINE: {short_name}")
print(f"{'#'*60}")

model, tokenizer = load_model_and_tokenizer(model_name)

start = time.time()
preds, raws = run_inference(model, tokenizer, baseline_test, batch_desc=f"Baseline {short_name}")
elapsed = time.time() - start

metrics = evaluate_predictions(true_labels_baseline, preds, short_name, "Baseline")
if metrics:
    metrics["time_seconds"] = elapsed
    all_results[f"{short_name}_baseline"] = metrics

print(f"\n--- Sample outputs ({short_name} baseline) ---")
for i in range(3):
    print(f"\nTrue: {true_labels_baseline[i]} | Predicted: {preds[i]}")
    print(f"Output: {raws[i][:300]}...")

del model, tokenizer
cleanup()


############################################################
# BASELINE: Llama-3.2-3B
############################################################


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

  Baseline Llama-3.2-3B: 0/300...
  Baseline Llama-3.2-3B: 50/300...
  Baseline Llama-3.2-3B: 100/300...
  Baseline Llama-3.2-3B: 150/300...
  Baseline Llama-3.2-3B: 200/300...
  Baseline Llama-3.2-3B: 250/300...

📊 Llama-3.2-3B — Baseline
  Accuracy:  0.5093
  Precision: 0.8571
  Recall:    0.4773
  F1 Score:  0.6131
  Unknown:   192/300

              precision    recall  f1-score   support

      benign     0.2203    0.6500    0.3291        20
    phishing     0.8571    0.4773    0.6131        88

    accuracy                         0.5093       108
   macro avg     0.5387    0.5636    0.4711       108
weighted avg     0.7392    0.5093    0.5605       108


--- Sample outputs (Llama-3.2-3B baseline) ---

True: benign | Predicted: benign
Output: Analysis:
- The URL structure appears to be legitimate, with a clear scheme, hostname, and registered domain. However, the redirect chain indicates that the final URL is not the original domain, but rather a different domain (useluckyplay-fa

In [ ]:
short_name = "Llama-3.1-8B"
model_name = "meta-llama/Llama-3.1-8B-Instruct"

print(f"\n{'#'*60}")
print(f"# BASELINE: {short_name}")
print(f"{'#'*60}")

model, tokenizer = load_model_and_tokenizer(model_name)

start = time.time()
preds, raws = run_inference(model, tokenizer, baseline_test, batch_desc=f"Baseline {short_name}")
elapsed = time.time() - start

metrics = evaluate_predictions(true_labels_baseline, preds, short_name, "Baseline")
if metrics:
    metrics["time_seconds"] = elapsed
    all_results[f"{short_name}_baseline"] = metrics

print(f"\n--- Sample outputs ({short_name} baseline) ---")
for i in range(3):
    print(f"\nTrue: {true_labels_baseline[i]} | Predicted: {preds[i]}")
    print(f"Output: {raws[i][:300]}...")

del model, tokenizer
cleanup()


############################################################
# BASELINE: Llama-3.1-8B
############################################################


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

  Baseline Llama-3.1-8B: 0/300...
  Baseline Llama-3.1-8B: 50/300...
  Baseline Llama-3.1-8B: 100/300...
  Baseline Llama-3.1-8B: 150/300...
  Baseline Llama-3.1-8B: 200/300...
  Baseline Llama-3.1-8B: 250/300...

📊 Llama-3.1-8B — Baseline
  Accuracy:  0.7559
  Precision: 0.9740
  Recall:    0.7212
  F1 Score:  0.8287
  Unknown:   173/300

              precision    recall  f1-score   support

      benign     0.4200    0.9130    0.5753        23
    phishing     0.9740    0.7212    0.8287       104

    accuracy                         0.7559       127
   macro avg     0.6970    0.8171    0.7020       127
weighted avg     0.8737    0.7559    0.7828       127


--- Sample outputs (Llama-3.1-8B baseline) ---

True: benign | Predicted: phishing
Output: Analysis:
- The URL's registered domain "7k4695.casino" is not a well-known brand, but it is similar to a known brand "7K casino" mentioned in the page title.
- The URL's hostname and registered domain are the same, indicating a direct dom

In [16]:
short_name = "Llama-3.2-3B"
model_name = "meta-llama/Llama-3.2-3B-Instruct"

model, tokenizer, train_time = fine_tune_model(model_name, short_name, train_examples)

true_labels_test = [ex["label"] for ex in test_examples]
start = time.time()
preds, raws = run_inference(model, tokenizer, test_examples, batch_desc=f"Fine-tuned {short_name}")
eval_time = time.time() - start

metrics = evaluate_predictions(true_labels_test, preds, short_name, "Fine-tuned")
if metrics:
    metrics["train_time_min"] = train_time / 60
    metrics["eval_time_seconds"] = eval_time
    all_results[f"{short_name}_finetuned"] = metrics

print(f"\n--- Sample outputs ({short_name} fine-tuned) ---")
for i in range(3):
    print(f"\nTrue: {true_labels_test[i]} | Predicted: {preds[i]}")
    print(f"Output: {raws[i][:500]}...")

del model, tokenizer
cleanup()


############################################################
# FINE-TUNING: Llama-3.2-3B
############################################################


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


  Total params:     3,237,063,680
  Trainable params:   24,313,856 (0.75%)


Adding EOS to train dataset:   0%|          | 0/18000 [00:00<?, ? examples/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.



🚀 Training Llama-3.2-3B...


Step,Training Loss
50,1.910695
100,1.122239
150,0.971023
200,0.929245
250,1.027878
300,3.900335
350,6.772737
400,6.363192
450,6.277574
500,6.286526


✅ Llama-3.2-3B training complete in 316.6 minutes
  Fine-tuned Llama-3.2-3B: 0/2000...
  Fine-tuned Llama-3.2-3B: 50/2000...
  Fine-tuned Llama-3.2-3B: 100/2000...
  Fine-tuned Llama-3.2-3B: 150/2000...
  Fine-tuned Llama-3.2-3B: 200/2000...
  Fine-tuned Llama-3.2-3B: 250/2000...
  Fine-tuned Llama-3.2-3B: 300/2000...
  Fine-tuned Llama-3.2-3B: 350/2000...
  Fine-tuned Llama-3.2-3B: 400/2000...
  Fine-tuned Llama-3.2-3B: 450/2000...


KeyboardInterrupt: 

In [ ]:
short_name = "Llama-3.1-8B"
model_name = "meta-llama/Llama-3.1-8B-Instruct"

model, tokenizer, train_time = fine_tune_model(model_name, short_name, train_examples)

true_labels_test = [ex["label"] for ex in test_examples]
start = time.time()
preds, raws = run_inference(model, tokenizer, test_examples, batch_desc=f"Fine-tuned {short_name}")
eval_time = time.time() - start

metrics = evaluate_predictions(true_labels_test, preds, short_name, "Fine-tuned")
if metrics:
    metrics["train_time_min"] = train_time / 60
    metrics["eval_time_seconds"] = eval_time
    all_results[f"{short_name}_finetuned"] = metrics

print(f"\n--- Sample outputs ({short_name} fine-tuned) ---")
for i in range(3):
    print(f"\nTrue: {true_labels_test[i]} | Predicted: {preds[i]}")
    print(f"Output: {raws[i][:500]}...")

del model, tokenizer
cleanup()


############################################################
# FINE-TUNING: Llama-3.1-8B
############################################################


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


  Total params:     8,072,204,288
  Trainable params:   41,943,040 (0.52%)


Adding EOS to train dataset:   0%|          | 0/18000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/18000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.



🚀 Training Llama-3.1-8B...


Step,Training Loss
50,1.582416
100,0.912020
150,0.865027
200,1.364848
250,4.189093
300,6.480237
350,6.602579
400,6.413818
450,6.109271
500,5.813723


In [ ]:
results_df = pd.DataFrame(all_results).T
print("\n" + "="*80)
print("📊 FULL RESULTS COMPARISON")
print("="*80)
print(results_df[["model", "phase", "accuracy", "precision", "recall", "f1", "unknown"]].to_string())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.dpi'] = 120

# Prepare data for plotting
models_list = list(MODELS.keys())
metrics_names = ["accuracy", "precision", "recall", "f1"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, short_name in enumerate(models_list):
    ax = axes[idx]
    baseline_key = f"{short_name}_baseline"
    finetuned_key = f"{short_name}_finetuned"

    baseline_vals = [all_results.get(baseline_key, {}).get(m, 0) for m in metrics_names]
    finetuned_vals = [all_results.get(finetuned_key, {}).get(m, 0) for m in metrics_names]

    x = np.arange(len(metrics_names))
    width = 0.35

    bars1 = ax.bar(x - width/2, baseline_vals, width, label="Baseline", color="#FF6B6B", alpha=0.85)
    bars2 = ax.bar(x + width/2, finetuned_vals, width, label="Fine-tuned", color="#4ECDC4", alpha=0.85)

    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title(f"{short_name}", fontsize=14, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([m.capitalize() for m in metrics_names])
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax.annotate(f"{height:.3f}",
                           xy=(bar.get_x() + bar.get_width()/2, height),
                           xytext=(0, 3), textcoords="offset points",
                           ha="center", va="bottom", fontsize=8)

plt.suptitle("Phishing Detection: Baseline vs Fine-tuned", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()
print("📊 Chart saved to Drive!")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

labels = []
f1_scores = []
colors = []

for short_name in models_list:
    for phase, color in [("baseline", "#FF6B6B"), ("finetuned", "#4ECDC4")]:
        key = f"{short_name}_{phase}"
        if key in all_results:
            labels.append(f"{short_name}\n({phase.capitalize()})")
            f1_scores.append(all_results[key]["f1"])
            colors.append(color)

bars = ax.bar(labels, f1_scores, color=colors, alpha=0.85, edgecolor="white", linewidth=1.5)

for bar, score in zip(bars, f1_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{score:.4f}", ha="center", va="bottom", fontweight="bold", fontsize=11)

ax.set_ylim(0, 1.1)
ax.set_ylabel("F1 Score", fontsize=12)
ax.set_title("F1 Score Comparison: Baseline vs Fine-tuned", fontsize=14, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
ax.axhline(y=0.95, color="gray", linestyle="--", alpha=0.5, label="0.95 target")
ax.legend()

plt.tight_layout()
plt.show()
print("📊 F1 chart saved to Drive!")